In [2]:
import numpy as np
import pandas as pd
import nltk

In [5]:
amazon_df = pd.read_csv('amazon_product.csv')
amazon_df.head()

,id,Title,Description,Category
0,1,Swissmar Capstore Select Storage Rack for 18-...,Swissmar's capstore select 18 storage unit kee...,Home & Kitchen Kitchen & Dining Kitchen Utens...
1,2,Gemini200 Delta CV-880 Gold Crown Livery Airc...,Welcome to the exciting world of GeminiJets! O...,Toys & Games Hobbies Models & Model Kits Pre-...
2,5,Superior Threads 10501-2172 Magnifico Cream P...,"For quilting and embroidery, this product is m...","Arts, Crafts & Sewing Sewing Thread & Floss S..."
3,6,Fashion Angels Color Rox Hair Chox Kit,Experiment with the haute trend of hair chalki...,Beauty & Personal Care Hair Care Hair Colorin...
4,8,Union Creative Giant Killing Figure 05: Daisu...,From Union Creative. Turn your display shelf i...,Toys & Games › Action Figures & Statues › Sta...


In [6]:
amazon_df.drop('id',axis=1,inplace=True)

In [7]:
amazon_df.head()

,Title,Description,Category
0,Swissmar Capstore Select Storage Rack for 18-...,Swissmar's capstore select 18 storage unit kee...,Home & Kitchen Kitchen & Dining Kitchen Utens...
1,Gemini200 Delta CV-880 Gold Crown Livery Airc...,Welcome to the exciting world of GeminiJets! O...,Toys & Games Hobbies Models & Model Kits Pre-...
2,Superior Threads 10501-2172 Magnifico Cream P...,"For quilting and embroidery, this product is m...","Arts, Crafts & Sewing Sewing Thread & Floss S..."
3,Fashion Angels Color Rox Hair Chox Kit,Experiment with the haute trend of hair chalki...,Beauty & Personal Care Hair Care Hair Colorin...
4,Union Creative Giant Killing Figure 05: Daisu...,From Union Creative. Turn your display shelf i...,Toys & Games › Action Figures & Statues › Sta...


In [8]:
amazon_df.isnull().sum()

Title          0
Description    0
Category       0
dtype: int64

In [21]:
from nltk.stem.snowball import SnowballStemmer
from nltk.tokenize import word_tokenize
nltk.download('punkt_tab')
stemmer = SnowballStemmer('english')
def tokenize_stem(text):
    tokens = nltk.word_tokenize(text.lower()) #seperates words as single unit
    stemmed = [stemmer.stem(w) for w in tokens] #converts different forms of a word to single
    return " ".join(stemmed)
#this function first converts the word into lowercase, then word tokenize it, then removes the extra forms of the word and return best form(eg. love for lover, loved)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [22]:
amazon_df['stemmed_tokens'] = amazon_df.apply(lambda row:tokenize_stem(row['Title']+" "+row['Description']),axis=1)

In [23]:
amazon_df.head()

,Title,Description,Category,stemmed_tokens
0,Swissmar Capstore Select Storage Rack for 18-...,Swissmar's capstore select 18 storage unit kee...,Home & Kitchen Kitchen & Dining Kitchen Utens...,swissmar capstor select storag rack for 18-pac...
1,Gemini200 Delta CV-880 Gold Crown Livery Airc...,Welcome to the exciting world of GeminiJets! O...,Toys & Games Hobbies Models & Model Kits Pre-...,gemini200 delta cv-880 gold crown liveri aircr...
2,Superior Threads 10501-2172 Magnifico Cream P...,"For quilting and embroidery, this product is m...","Arts, Crafts & Sewing Sewing Thread & Floss S...",superior thread 10501-2172 magnifico cream puf...
3,Fashion Angels Color Rox Hair Chox Kit,Experiment with the haute trend of hair chalki...,Beauty & Personal Care Hair Care Hair Colorin...,fashion angel color rox hair chox kit experi w...
4,Union Creative Giant Killing Figure 05: Daisu...,From Union Creative. Turn your display shelf i...,Toys & Games › Action Figures & Statues › Sta...,union creativ giant kill figur 05 : daisuk tsu...


In [45]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
tfidv = TfidfVectorizer(tokenizer = tokenize_stem)
def cosine_sim(txt1,txt2):
    matrix = tfidv.fit_transform([txt1,txt2])
    return cosine_similarity(matrix)[0][1]

In [46]:
def search_product(query):
    stemmed_query = tokenize_stem(query)
    #calculating cosine similarity between query and stemmed token columns
    amazon_df['similarity'] = amazon_df['stemmed_tokens'].apply(lambda x:cosine_sim(stemmed_query,x))
    result = amazon_df.sort_values(by=['similarity'],ascending=False).head(10)[['Title','Description','Category']]
    return result

In [32]:
amazon_df['Title'][0]

' Swissmar Capstore Select Storage Rack for 18-Pack '

In [47]:
search_product(' Swissmar Capstore Select Storage Rack for 18-Pack ')

,Title,Description,Category
475,Pacon Spectra Glitter Sparkling Crystals,"Change all to: Spectra Glitter, Sparkling Crys...","Arts, Crafts & Sewing › Crafting › Craft Supp..."
463,Versio Mobile 3-Pack Screen Protector for LG ...,Clear 3 Pack Screen Protector For LG G2,Cell Phones & Accessories Accessories Screen ...
593,Qmadix QM-SPHTC6410 Screen Protector for HTC ...,"Protect your device's display from dust, scrat...",Cell Phones & Accessories Accessories Screen ...
526,"Titan Classic Spandex Dreadlock Cap, Regular",Spandex Dreadlock Cap - Regular Size. Covers M...,Beauty & Personal Care › Hair Care
177,Sargent Art 22-7099 6-Count 8-Ounce Fluoresce...,Sargent art's washable watercolor magic is uni...,"Arts, Crafts & Sewing › Painting, Drawing & A..."
0,Swissmar Capstore Select Storage Rack for 18-...,Swissmar's capstore select 18 storage unit kee...,Home & Kitchen Kitchen & Dining Kitchen Utens...
147,Martha Stewart Crafts 32802 Stencil Brushes (...,Transfer patterns with ease by painting over t...,"Arts, Crafts & Sewing Crafting Paper & Paper ..."
110,Core Products Elastic Criss Cross Back Suppor...,The Elastic Crisscross Back Support helps reli...,Health & Household › Medical Supplies & Equip...
384,Visol Ferrara Red Cigar Case - Holds 2-3 Cigars,"Vibrant red, soft leather wraps this crushproo...",Health & Household Household Supplies Tobacco...
13,BRUT After Shave Classic Fragrance 5 oz (Pack...,BRUT After Shave Classic Fragrance 5 oz (Pack ...,Beauty & Personal Care › Shave & Hair Removal...


In [48]:
amazon_df['Title'][11]

' Gum Power Rangers Timer Light Toothbrush - Soft (3 Pack) '

In [49]:
search_product(' Gum Power Rangers Timer Light Toothbrush - Soft (3 Pack) ')

C:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\feature_extraction\text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,Title,Description,Category
11,Gum Power Rangers Timer Light Toothbrush - So...,Plaque and cavities don’t stand a chance with ...,Beauty & Personal Care › Oral Care › Children...
535,MixBin Wireless Bluetooth Speaker with Suctio...,Be the life of the party with this Bluetooth s...,Electronics › Portable Audio & Video › Portab...
221,Flat Angled Brush ABT Animal-Free Eye Shadow ...,"To get precisely the look you want, you need a...",Beauty & Personal Care Tools & Accessories Ma...
265,"Rikki Knight Letter""T"" Blue Houndstooth Monog...","Whether for the home or the office, the Letter...",Office Products › Office & School Supplies › ...
338,PuTwo Makeup Brush Holder Dustproof Storage B...,PuTwo Makeup Brush Holder Dustproof Storage Bo...,Beauty & Personal Care Tools & Accessories Ba...
138,Power Systems Single Premium Hanging Club Mat,A Power Systems Top Seller! These mats offer t...,Sports & Outdoors Sports & Fitness Exercise &...
166,Dimension - A 3D Fast-Paced Puzzle Game from ...,"Dimension is a fast-paced, innovative puzzle g...",Toys & Games › Puzzles
147,Martha Stewart Crafts 32802 Stencil Brushes (...,Transfer patterns with ease by painting over t...,"Arts, Crafts & Sewing Crafting Paper & Paper ..."
533,SilverStone Technology Smart Four Port USB 3....,To satisfy users' increasing sophisticated nee...,Cell Phones & Accessories Accessories Batteri...
591,FILOFAX Portable Hole Punch for Personal & Pe...,The Filofax Hole Punch offers a great solution...,Tools & Home Improvement Power & Hand Tools H...
